<a href="https://colab.research.google.com/github/quirocode/Medical-data-privacy-attack/blob/main/notebooks/linkage_attack_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Cloud Data Ingestion & Environmental Setup

In [ ]:
# 1. Configuración de la API Token de Kaggle (Método directo)
import os
API_TOKEN = "KGAT_950ef98680cbc7df25e0d5ae167e06a7"

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(API_TOKEN)
os.chmod('/root/.kaggle/access_token', 0o600)

# 2. Descarga y descompresión del dataset masivo de Synthea
!kaggle datasets download -d drscarlat/syntheacovid100k
!unzip -q syntheacovid100k.zip -d synthea_data
print("¡Pipeline de datos inicializado: Dataset de Synthea descargado!")

## 2. Relational Data Wrangling & Statistical Imputation

In [ ]:
import pandas as pd

# Buscar rutas dinámicas de los archivos descompresionados
ruta_pacientes, ruta_condiciones = "", ""
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.lower() == 'patients.csv':
            ruta_pacientes = os.path.join(root, file)
        elif file.lower() == 'conditions.csv':
            ruta_condiciones = os.path.join(root, file)

# Cargar y renombrar llaves relacionales
df_patients = pd.read_csv(ruta_pacientes)
df_conditions = pd.read_csv(ruta_condiciones)
df_patients = df_patients.rename(columns={'Id': 'PATIENT'})

# Selección de variables demográficas y clínicas
df_patients = df_patients[['PATIENT', 'FIRST', 'LAST', 'BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL', 'ETHNICITY', 'CITY']]
df_conditions = df_conditions[['PATIENT', 'DESCRIPTION']]
df_master = pd.merge(df_patients, df_conditions, on='PATIENT', how='inner')

# Imputación de datos faltantes (Data Imputation)
qis = ['BIRTHDATE', 'GENDER', 'ZIP', 'RACE', 'MARITAL', 'ETHNICITY', 'CITY']
for qi in qis:
    df_master[qi] = df_master[qi].fillna('Unknown')

df_master['Nombre_Real'] = df_master['FIRST'].astype(str) + ' ' + df_master['LAST'].astype(str)
print(f"Base maestra unificada y consolidada con {len(df_master)} registros médicos.")

## 3. Asymmetric Multi-dimensional Linkage Attack Execution

In [ ]:
# Construcción de los escenarios (Padrón Público vs Base de Hospital)
public_registry = df_master[['Nombre_Real'] + qis].copy().drop_duplicates()
hospital_data = df_master[qis + ['DESCRIPTION']].copy()

print("Ejecutando cruce determinista asimétrico sobre los cuasi-identificadores...")
# Identificar registros únicos en la población civil
public_unicos = public_registry.drop_duplicates(subset=qis, keep=False)

# Mapeo y re-identificación final
ataque_exitoso = pd.merge(hospital_data, public_unicos, on=qis, how='inner')

## 4. Vulnerability Metrics & Privacy Risk Evaluation

In [ ]:
# Cálculo exacto de la tasa de vulnerabilidad
tasa_exito = (len(ataque_exitoso) / len(df_master)) * 100

print("\n" + "="*70)
print("🔬 RESULTADOS DEL LÍMITE TEÓRICO (BIG DATA) - COMPORTAMIENTO DE PRIVACIDAD")
print("="*70)
print(f"Población clínica total analizada:  {len(df_master)} registros.")
print(f"Historiales médicos vulnerados:     {len(ataque_exitoso)}")
print(f"TASA MÁXIMA DE RE-IDENTIFICACIÓN:   {tasa_exito:.2f}%")
print("-" * 70)
print(ataque_exitoso[['Nombre_Real', 'DESCRIPTION', 'CITY', 'RACE', 'MARITAL']].head())